[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [sqlite3, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)

# Joins


## What you will be able to do

Combine the rows of two tables with an inner, left, right, full or cross join, or join a table to
itself, and say before running the query what the result will do with a row that has no partner.
Write a full join where `FULL JOIN` is missing, find the rows with no match, and keep a join from
counting a row twice.


## The idea

### Joins at a glance

A summary to come back to, before any of the detail.

#### The joins

| Join type | What it returns | Handling of unmatched rows |
|---|---|---|
| `INNER JOIN` | only the rows with a match in both tables | left out entirely |
| `LEFT JOIN` | every row of the left table, and the matching rows of the right | the right table's columns filled with `NULL` |
| `RIGHT JOIN` | every row of the right table, and the matching rows of the left | the left table's columns filled with `NULL` |
| `FULL JOIN` | every row of both tables | the missing table's columns filled with `NULL` |
| `CROSS JOIN` | every combination of a row from one table with a row from the other | no matching is done, so no row is unmatched |
| self join | rows of a table matched with rows of the same table | as the `INNER` or `LEFT` join it is written with |

The left table is the one named before the join, and the right table the one named after it.
`JOIN` alone means `INNER JOIN`, and `OUTER` is optional, so `LEFT OUTER JOIN`, `RIGHT OUTER JOIN`
and `FULL OUTER JOIN` are the joins above. SQLite has had `RIGHT` and `FULL` joins since version
3.39.0, from 2022.

#### The tables every example uses

`Employees`, where `DeptID` names an employee's department and `ManagerID` the `EmpID` of their
manager:

| EmpID | Name | DeptID | ManagerID | Salary |
|---|---|---|---|---|
| 1 | Alice | 1 | `NULL` | 72000 |
| 2 | Bob | 2 | 1 | 64000 |
| 3 | Charlie | `NULL` | 1 | 51000 |

`Departments`:

| DeptID | DeptName |
|---|---|
| 1 | HR |
| 2 | IT |
| 3 | Marketing |

Charlie has no department yet, and Marketing has no employees.

### The problem

A database keeps a fact in one place. An employee's department is stored as a number, `DeptID`, and
the department's name only in `Departments`, so a question as plain as which department every
employee is in needs both tables, and a join puts their rows back together.

The difficult rows are the ones with no partner. Charlie has no department, and Marketing has no
employees. An inner join drops both without a word, so a head count built on it leaves out a new
starter, and the wrong outer join lists a department nobody works in as though someone did. A join
can also match a row more than once and repeat it, and a sum over the repeated rows comes out too
big, with nothing to say so. Which join to write depends on which of those rows the answer needs,
and every one of these mistakes returns a result that looks right.

### What a join is

> A **join** combines the rows of two tables into the rows of one result, pairing a row of one with
> a row of the other. Its **`ON`** condition says which pairs belong together. An **inner join**
> returns only the pairs for which the condition is true, and an **outer join**, `LEFT`, `RIGHT` or
> `FULL`, also returns the rows of one table or both that found no partner, with `NULL` in the other
> table's columns. A **cross join** has no condition and returns every pair. A **self join** is a
> join between a table and itself, whose two copies are told apart by **aliases**.

### Why it works that way

- **A join starts from every pair.** In principle, a join pairs every row of the left table with
  every row of the right, which is a cross join, and keeps the pairs for which `ON` is true. Three
  employees and three departments make nine pairs, and two of them match.
- **`NULL` matches nothing.** `NULL = 1` is not true, and neither is `NULL = NULL`, so Charlie's
  missing `DeptID` pairs with no department, whatever the departments hold.
- **An outer join adds back what matched nothing.** A left join returns the inner join's rows and
  every left row that found no partner, filled out with `NULL`. A right join does the same for the
  right table, and a full join for both.
- **Left and right are only the order of the tables.** `Employees RIGHT JOIN Departments` returns the
  rows of `Departments LEFT JOIN Employees`, which is why SQLite managed without `RIGHT JOIN` until
  version 3.39.0, and why a lot of SQL is written with `LEFT JOIN` alone.
- **A join repeats a row once for every match.** A manager of two people appears in two rows of a
  join from employees to their managers, so a `SUM` over that join counts the manager's salary twice.

### Where this shows up

Every relational database writes these joins the same way, except that MySQL has no `FULL JOIN`, and
PostgreSQL, in the **asyncpg and psycopg3, Deep Dive** guide, has every join here. The **Pandas,
Deep Dive** guide joins DataFrames with `merge`, whose `how` can be `'inner'`, `'left'`, `'right'`,
`'outer'` or `'cross'`, and the **Polars, Deep Dive** guide's `join` offers the same choices. The
**SQLAlchemy, Deep Dive** guide writes `join` and `outerjoin`, and works out `ON` from the foreign
keys a model declares. In this guide, every query that puts a station's name beside its readings is
a join, as the **Tables and Queries** notebook first showed.

### What this notebook covers

- The two tables, and the rows in them with no partner
- `INNER JOIN`, and why Charlie and Marketing are left out
- `LEFT JOIN`, with `NULL` for Charlie's department
- `RIGHT JOIN`, and the `LEFT JOIN` that returns the same rows
- `FULL JOIN`, and the same rows with `UNION ALL` where `FULL JOIN` is missing
- `CROSS JOIN`, and every join as the pairs a cross join makes, filtered by `ON`
- Self joins: employees and their managers, and pairs of employees
- Three tables in one query
- Rows with no match, with `LEFT JOIN` and `IS NULL`, and with `NOT EXISTS`
- A condition in `ON` against the same condition in `WHERE`, and `USING`
- When to use which join
- A head count and payroll for every department, and for the employees with none
- Seven errors: a self join with no aliases, a table's name after its alias, a `JOIN` with no `ON`,
  `COUNT(*)` after a left join, a salary counted twice, `NOT IN` beside `NULL`, and a condition on
  the left table in `ON`

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things, and the rest of the notebook takes this apart piece by piece.

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE Departments (DeptID INTEGER PRIMARY KEY, DeptName TEXT NOT NULL);
    CREATE TABLE Employees (EmpID INTEGER PRIMARY KEY, Name TEXT NOT NULL, DeptID INTEGER);
    INSERT INTO Departments VALUES (1, 'HR'), (2, 'IT'), (3, 'Marketing');
    INSERT INTO Employees VALUES (1, 'Alice', 1), (2, 'Bob', 2), (3, 'Charlie', NULL);
""")

for join in ["INNER JOIN", "LEFT JOIN"]:
    rows = conn.execute(f"""
        SELECT Employees.Name, Departments.DeptName
        FROM Employees
        {join} Departments ON Employees.DeptID = Departments.DeptID
        ORDER BY Employees.Name
    """).fetchall()
    print(f"{join:<10}", rows)
conn.close()
```

```
INNER JOIN [('Alice', 'HR'), ('Bob', 'IT')]
LEFT JOIN  [('Alice', 'HR'), ('Bob', 'IT'), ('Charlie', None)]
```

The same query twice, with only the join changed. The inner join returned the two employees whose
`DeptID` matches a department. The left join kept Charlie as well, with `None`, which is how Python
shows `NULL`, for a department. Marketing, which no employee matched, is in neither.


## Setup

One import, and the two tables every example uses, built in a database in memory, since nothing here
needs to outlast the notebook.

- `sqlite3` builds the tables and runs every join

`show` runs a query and prints its rows as a table, with `NULL` where a value is missing, as SQL
writes it. `RIGHT_AND_FULL` records whether this SQLite has `RIGHT` and `FULL` joins, which arrived
in version 3.39.0. The cells that use them run a query that returns the same rows on an older SQLite,
so every cell prints the same wherever it runs.


In [1]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.executescript("""
    CREATE TABLE Departments (
        DeptID   INTEGER PRIMARY KEY,
        DeptName TEXT NOT NULL
    );
    CREATE TABLE Employees (
        EmpID     INTEGER PRIMARY KEY,
        Name      TEXT NOT NULL,
        DeptID    INTEGER REFERENCES Departments (DeptID),
        ManagerID INTEGER REFERENCES Employees (EmpID),
        Salary    INTEGER NOT NULL
    );
    INSERT INTO Departments (DeptID, DeptName) VALUES (1, 'HR'), (2, 'IT'), (3, 'Marketing');
    INSERT INTO Employees (EmpID, Name, DeptID, ManagerID, Salary) VALUES
        (1, 'Alice', 1, NULL, 72000),
        (2, 'Bob', 2, 1, 64000),
        (3, 'Charlie', NULL, 1, 51000);
""")
RIGHT_AND_FULL = sqlite3.sqlite_version_info >= (3, 39, 0)


def show(sql):
    """Run a query and print its rows as a table, with NULL for a missing value."""
    cursor = conn.execute(sql)
    names = [column[0] for column in cursor.description]
    rows = [["NULL" if value is None else str(value) for value in row] for row in cursor]
    widths = [max(len(text) for text in column) for column in zip(names, *rows)]
    for line in [names, ["-" * width for width in widths], *rows]:
        print("  ".join(text.ljust(width) for text, width in zip(line, widths)).rstrip())
    print(f"({len(rows)} row{'' if len(rows) == 1 else 's'})")


print("tables:", [name for (name,) in conn.execute("SELECT name FROM sqlite_schema WHERE type = 'table' ORDER BY name")])


tables: ['Departments', 'Employees']


## Worked examples

### The two tables

Both tables, in full, before any join:


In [2]:
show("SELECT * FROM Employees ORDER BY EmpID")
print()
show("SELECT * FROM Departments ORDER BY DeptID")


EmpID  Name     DeptID  ManagerID  Salary
-----  -------  ------  ---------  ------
1      Alice    1       NULL       72000
2      Bob      2       1          64000
3      Charlie  NULL    1          51000
(3 rows)

DeptID  DeptName
------  ---------
1       HR
2       IT
3       Marketing
(3 rows)


Charlie's `DeptID` is `NULL`, since Charlie has no department yet, and no employee's `DeptID` is 3,
so Marketing has no employees. Those two rows, one in each table with no partner in the other, are
the rows the joins below treat differently. Alice, who manages Bob and Charlie, has no manager, so
Alice's `ManagerID` is `NULL` too.

### INNER JOIN

An inner join returns the pairs of rows for which `ON` is true. Here the condition is
`Employees.DeptID = Departments.DeptID`, which pairs an employee with the department their `DeptID`
names:


In [3]:
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    INNER JOIN Departments ON Employees.DeptID = Departments.DeptID
    ORDER BY Employees.Name
""")


Name   DeptName
-----  --------
Alice  HR
Bob    IT
(2 rows)


Alice and Bob found their departments. Charlie's `DeptID` is `NULL`, which equals no `DeptID`, so
Charlie is left out, and no employee's `DeptID` is 3, so Marketing is left out too. `JOIN` on its own
means `INNER JOIN`. Every query in this notebook ends with `ORDER BY`, since SQL promises no order of
rows without one.

### LEFT JOIN

A left join returns every row of the left table, the one named before `LEFT JOIN`, with its matching
rows from the right table:


In [4]:
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    LEFT JOIN Departments ON Employees.DeptID = Departments.DeptID
    ORDER BY Employees.Name
""")


Name     DeptName
-------  --------
Alice    HR
Bob      IT
Charlie  NULL
(3 rows)


Every employee is there. Charlie matched no department, so the column from `Departments` came back as
`NULL`. `LEFT OUTER JOIN` is the same join, since `OUTER` is optional. Which table is the left one is
decided by the order in `FROM`, never by which side of `ON` a column is written on.

### RIGHT JOIN

A right join keeps every row of the right table instead, the one named after `RIGHT JOIN`. The
`NULLS LAST` in `ORDER BY` puts a row with no name at the end, where SQLite would otherwise sort it
first:


In [5]:
right_join = """
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    RIGHT JOIN Departments ON Employees.DeptID = Departments.DeptID
    ORDER BY Employees.Name NULLS LAST
"""
the_same_rows = """
    SELECT Employees.Name, Departments.DeptName
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    ORDER BY Employees.Name NULLS LAST
"""
show(right_join if RIGHT_AND_FULL else the_same_rows)
print("the same rows, from a left join with the tables swapped:")
show(the_same_rows)
print("and without NULLS LAST:")
show(the_same_rows.replace(" NULLS LAST", ""))


Name   DeptName
-----  ---------
Alice  HR
Bob    IT
NULL   Marketing
(3 rows)
the same rows, from a left join with the tables swapped:
Name   DeptName
-----  ---------
Alice  HR
Bob    IT
NULL   Marketing
(3 rows)
and without NULLS LAST:
Name   DeptName
-----  ---------
NULL   Marketing
Alice  HR
Bob    IT
(3 rows)


Every department is there, and Marketing, which no employee matched, has `NULL` for a name. Charlie
is gone again, since a right join adds the unmatched rows of the right table only. `the_same_rows`
swaps the two tables and writes a left join, and returns exactly the same rows, printed under them,
which is what the cell's first line runs on a version of SQLite older than 3.39.0, where there is no
`RIGHT JOIN`. The third query is the second without `NULLS LAST`, where Marketing's `NULL` name sorts
to the top instead.

### FULL JOIN

A full join keeps the unmatched rows of both tables:


In [6]:
full_join = """
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    FULL JOIN Departments ON Employees.DeptID = Departments.DeptID
    ORDER BY Employees.Name NULLS LAST
"""
without_full_join = """
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    LEFT JOIN Departments ON Employees.DeptID = Departments.DeptID
    UNION ALL
    SELECT Employees.Name, Departments.DeptName
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    WHERE Employees.EmpID IS NULL
    ORDER BY Name NULLS LAST
"""
show(full_join if RIGHT_AND_FULL else without_full_join)


Name     DeptName
-------  ---------
Alice    HR
Bob      IT
Charlie  NULL
NULL     Marketing
(4 rows)


Four rows: the two matches, Charlie with no department, and Marketing with no employee. Written with
`OUTER`, as `FULL OUTER JOIN`, it is the same join. On a version of SQLite older than 3.39.0 the
cell runs `without_full_join`, which the next example takes apart.

### A full join without FULL JOIN

MySQL has no `FULL JOIN`, and SQLite had none before 3.39.0, so a full join is often written as two
left joins, one from each table:


In [7]:
show(without_full_join)


Name     DeptName
-------  ---------
Alice    HR
Bob      IT
Charlie  NULL
NULL     Marketing
(4 rows)


The same four rows. The first query is the left join from `Employees`, with every employee. The
second joins from `Departments` and keeps only the departments no employee matched, by testing
`Employees.EmpID IS NULL`: `EmpID` is the primary key, never `NULL` in a real row, so a `NULL` there
means the join filled the row in. `UNION ALL` returns the rows of both queries, which need the same
number of columns, and the one `ORDER BY` at the end sorts them all, by the first query's column
names.

### CROSS JOIN

A cross join has no `ON`. It pairs every row of one table with every row of the other:


In [8]:
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    CROSS JOIN Departments
    ORDER BY Employees.Name, Departments.DeptID
""")


Name     DeptName
-------  ---------
Alice    HR
Alice    IT
Alice    Marketing
Bob      HR
Bob      IT
Bob      Marketing
Charlie  HR
Charlie  IT
Charlie  Marketing
(9 rows)


Nine rows, three employees times three departments, with every employee beside every department
whatever their `DeptID`. A cross join is how to build every combination on purpose, such as a grid of
employees and departments for a sign-off sheet. `FROM Employees, Departments`, with a comma, returns
the same rows. SQLite gives `CROSS JOIN` a meaning of its own besides: its query planner never
reorders the two tables of a `CROSS JOIN`, where it is free to reorder the tables of an inner join.

### Every join is a cross join, filtered

The nine pairs again, with the `ON` condition of the joins above worked out for every pair, as 1 for
true, 0 for false, or `NULL`:


In [9]:
show("""
    SELECT Employees.Name,
           Employees.DeptID AS employee_dept,
           Departments.DeptID AS dept,
           Departments.DeptName,
           Employees.DeptID = Departments.DeptID AS on_is_true
    FROM Employees
    CROSS JOIN Departments
    ORDER BY Employees.Name, Departments.DeptID
""")


Name     employee_dept  dept  DeptName   on_is_true
-------  -------------  ----  ---------  ----------
Alice    1              1     HR         1
Alice    1              2     IT         0
Alice    1              3     Marketing  0
Bob      2              1     HR         0
Bob      2              2     IT         1
Bob      2              3     Marketing  0
Charlie  NULL           1     HR         NULL
Charlie  NULL           2     IT         NULL
Charlie  NULL           3     Marketing  NULL
(9 rows)


The inner join returned the two pairs where `on_is_true` is 1. A comparison with `NULL` is `NULL`,
neither true nor false, so no pair with Charlie is true, and a join keeps only true. The left join
then added Charlie, who has no true pair, the right join added Marketing, whose column holds no 1,
and the full join added both. SQLite does not build every pair to run a join: it uses the condition,
and an index where one exists, to go straight to the matches, as the **Indexes and Query Plans**
notebook shows.

### Self joins

A self join joins a table to itself. An employee's `ManagerID` is the `EmpID` of another row of
`Employees`, so finding a manager's name means reading `Employees` twice, as two copies with names of
their own: `e` for the employee and `m` for the manager. `AS` gives a table that alias, a second name
for the rest of the query:


In [10]:
for join in ["INNER JOIN", "LEFT JOIN"]:
    print(join)
    show(f"""
        SELECT e.Name AS employee, m.Name AS manager
        FROM Employees AS e
        {join} Employees AS m ON m.EmpID = e.ManagerID
        ORDER BY e.Name
    """)
    print()


INNER JOIN
employee  manager
--------  -------
Bob       Alice
Charlie   Alice
(2 rows)

LEFT JOIN
employee  manager
--------  -------
Alice     NULL
Bob       Alice
Charlie   Alice
(3 rows)



The inner join left out Alice, whose `ManagerID` is `NULL`, and the left join kept Alice, with `NULL`
for a manager, which is what the summary means by a self join depending on the join it is written
with. A self join cannot do without aliases, as a Common error shows.

A join's condition need not be equality. Every pair of employees, once:


In [11]:
show("""
    SELECT a.Name AS employee, b.Name AS colleague
    FROM Employees AS a
    JOIN Employees AS b ON a.EmpID < b.EmpID
    ORDER BY a.EmpID, b.EmpID
""")

for condition in ("a.EmpID < b.EmpID", "a.EmpID != b.EmpID", "1 = 1"):     # conditions this cell wrote
    pairs = conn.execute(f"SELECT COUNT(*) FROM Employees AS a JOIN Employees AS b ON {condition}").fetchone()[0]
    print(f"{condition:<18} {pairs} pairs")


employee  colleague
--------  ---------
Alice     Bob
Alice     Charlie
Bob       Charlie
(3 rows)
a.EmpID < b.EmpID  3 pairs
a.EmpID != b.EmpID 6 pairs
1 = 1              9 pairs


`a.EmpID < b.EmpID` keeps a pair only one way round, and never pairs an employee with the same
employee. The counts under the table are the same join with two other conditions: `!=` returns every
pair in both orders, and a condition true for every row returns all nine pairs, the cross join.

### Three tables in one query

Joins chain. A join adds a table to the rows made so far, and its `ON` can use any table named before
it. Every employee, with their department, their manager and their manager's department:


In [12]:
show("""
    SELECT e.Name AS employee, d.DeptName AS department, m.Name AS manager, md.DeptName AS manager_department
    FROM Employees AS e
    LEFT JOIN Departments AS d ON d.DeptID = e.DeptID
    LEFT JOIN Employees AS m ON m.EmpID = e.ManagerID
    LEFT JOIN Departments AS md ON md.DeptID = m.DeptID
    ORDER BY e.Name
""")


employee  department  manager  manager_department
--------  ----------  -------  ------------------
Alice     HR          NULL     NULL
Bob       IT          Alice    HR
Charlie   NULL        Alice    HR
(3 rows)


`Departments` appears twice, as `d` for the employee's department and `md` for the manager's.
Every join is a left join, since any inner join here would drop a row: joining `d` with `INNER JOIN`
would lose Charlie, and joining `m` with it would lose Alice.

### Rows with no match

The departments with no employees, found two ways. A left join fills an unmatched row with `NULL`, so
`WHERE` can keep only those rows, and `NOT EXISTS` asks the question directly:


In [13]:
show("""
    SELECT Departments.DeptName
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    WHERE Employees.EmpID IS NULL
    ORDER BY Departments.DeptName
""")
print()
show("""
    SELECT DeptName
    FROM Departments
    WHERE NOT EXISTS (SELECT 1 FROM Employees WHERE Employees.DeptID = Departments.DeptID)
    ORDER BY DeptName
""")


DeptName
---------
Marketing
(1 row)

DeptName
---------
Marketing
(1 row)


Both found Marketing. `EXISTS` is true when the query inside it returns any row, and that query can
name the outer query's table, here `Departments.DeptID`, so it is asked once for every department.
Keeping only the rows with no match is called an anti-join. Keeping the rows that have a match,
without repeating any of them, is a semi-join, which `EXISTS` without `NOT` writes.

### ON or WHERE, and USING

For an outer join, where a condition goes changes the answer. Every department, with the employees in
it who earn more than 70,000, first with the salary test in `ON`, then in `WHERE`:


In [14]:
print("in ON")
show("""
    SELECT Departments.DeptName, Employees.Name
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID AND Employees.Salary > 70000
    ORDER BY Departments.DeptID
""")
print()
print("in WHERE")
show("""
    SELECT Departments.DeptName, Employees.Name
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    WHERE Employees.Salary > 70000
    ORDER BY Departments.DeptID
""")


in ON
DeptName   Name
---------  -----
HR         Alice
IT         NULL
Marketing  NULL
(3 rows)

in WHERE
DeptName  Name
--------  -----
HR        Alice
(1 row)


In `ON`, the salary test decides which employees count as a match, so Bob no longer matches IT, and
IT stays, with `NULL`, as Marketing does. In `WHERE`, the test runs after the join and removes every
row whose `Salary` is not above 70,000, `NULL` included, which leaves only HR: the left join has
turned into an inner one. The **Tables and Queries** notebook met the same fault as a Common error.
For an inner join, the two places return the same rows.

When the column that links two tables has the same name in both, `USING` names it once, in place of
`ON`:


In [15]:
show("""
    SELECT *
    FROM Employees
    JOIN Departments USING (DeptID)
    ORDER BY EmpID
""")
show("""
    SELECT *
    FROM Employees
    JOIN Departments ON Employees.DeptID = Departments.DeptID
    ORDER BY EmpID
""")


EmpID  Name   DeptID  ManagerID  Salary  DeptName
-----  -----  ------  ---------  ------  --------
1      Alice  1       NULL       72000   HR
2      Bob    2       1          64000   IT
(2 rows)
EmpID  Name   DeptID  ManagerID  Salary  DeptID  DeptName
-----  -----  ------  ---------  ------  ------  --------
1      Alice  1       NULL       72000   1       HR
2      Bob    2       1          64000   2       IT
(2 rows)


`USING (DeptID)` means `ON Employees.DeptID = Departments.DeptID`, and `SELECT *` returned `DeptID`
once, where the same join written with `ON`, underneath, returns it twice, a column from each table.
A program reading rows by position would count the columns differently for the two.

### Which join

| Write | When | Why |
|---|---|---|
| `INNER JOIN` | only the rows with a partner belong in the answer | it drops unmatched rows from both tables, which is right when a row with no partner means nothing to the question |
| `LEFT JOIN` | every row of one table belongs in the answer, partner or not | it keeps the table named first complete, and runs on any version of SQLite |
| `RIGHT JOIN` | the table that has to be complete is named second, as in SQL written elsewhere | it returns the rows of a `LEFT JOIN` with the tables swapped, and needs SQLite 3.39.0 |
| `FULL JOIN` | unmatched rows from both tables belong in the answer, as when comparing two lists | it needs SQLite 3.39.0, and a `LEFT JOIN`, `UNION ALL` and a second `LEFT JOIN` return the same rows anywhere |
| `CROSS JOIN` | every combination is the answer | it applies no condition, so its rows number the product of the two tables' |
| a self join | rows of a table point at other rows of it, as `ManagerID` does | it reads the same table twice, with an alias for every copy |
| `NOT EXISTS` | the rows with no partner at all | it says what it means, and a `NULL` cannot trip it, where `NOT IN` can |

The default is `INNER JOIN` when every row the question cares about has a partner, and `LEFT JOIN`
from the table that has to be complete when some may not. A left join that returns more rows than
the inner join has found the rows with no partner.

### A head count and payroll for every department

The pieces of this notebook in one query: for every department, Marketing included, and for the
employees with no department, the number of people, the payroll, and who they are, with their
managers. A left join from `Departments` keeps every department, a self join finds every manager, and
`UNION ALL` adds a row for the employees a join from `Departments` cannot reach:


In [16]:
show("""
    SELECT d.DeptName AS department,
           COUNT(e.EmpID) AS people,
           coalesce(SUM(e.Salary), 0) AS payroll,
           group_concat(e.Name || ' (manager: ' || coalesce(m.Name, 'none') || ')', ', ') AS staff
    FROM Departments AS d
    LEFT JOIN Employees AS e ON e.DeptID = d.DeptID
    LEFT JOIN Employees AS m ON m.EmpID = e.ManagerID
    GROUP BY d.DeptID
    UNION ALL
    SELECT NULL,
           COUNT(e.EmpID),
           coalesce(SUM(e.Salary), 0),
           group_concat(e.Name || ' (manager: ' || coalesce(m.Name, 'none') || ')', ', ')
    FROM Employees AS e
    LEFT JOIN Employees AS m ON m.EmpID = e.ManagerID
    WHERE e.DeptID IS NULL
    ORDER BY department NULLS LAST
""")


department  people  payroll  staff
----------  ------  -------  ------------------------
HR          1       72000    Alice (manager: none)
IT          1       64000    Bob (manager: Alice)
Marketing   0       0        NULL
NULL        1       51000    Charlie (manager: Alice)
(4 rows)


Marketing has 0 people and a payroll of 0: `COUNT(e.EmpID)` counts only the rows a real employee
filled, and `coalesce` turns the `NULL` that `SUM` returns for no rows into 0. Marketing's staff is
`NULL`, since joining text to `NULL` with `||` gives `NULL`, and `group_concat` skips it. The last
row is everyone with no department. Every employee appears in exactly one row, so the payrolls add
up to the 187,000 the three salaries make, and a manager joined as `m` adds a name, never a salary.

### Where each part came from

| In the report | What it relies on | The section that showed it |
|---|---|---|
| `FROM Departments AS d LEFT JOIN Employees AS e` | every department, Marketing included | LEFT JOIN |
| `LEFT JOIN Employees AS m ON m.EmpID = e.ManagerID` | a second copy of `Employees`, for every employee's manager | Self joins |
| two joins in one `FROM` | a join adding a table to the rows made so far | Three tables in one query |
| `COUNT(e.EmpID)` and `coalesce(SUM(e.Salary), 0)` | aggregates that skip `NULL`, and a 0 for no rows | Aggregate functions, in the **SQL Syntax** notebook |
| `UNION ALL` and the query after it | the employees a join from `Departments` cannot reach | A full join without FULL JOIN |
| `ORDER BY department NULLS LAST` | the row with no department last | RIGHT JOIN |


## Your turn

Six tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than getting
there yourself, even slowly.

When you are ready: [**open the solutions notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/05-joins-solutions.ipynb).

**1.** Print every department with the names of the employees in it, Marketing included, using a
`LEFT JOIN` that names `Departments` first.


In [17]:
# your code here


**2.** Print the departments that have at least one employee, every department once, first with
`EXISTS` and then with an `INNER JOIN` and `DISTINCT`.


In [18]:
# your code here


**3.** Print every employee with their manager's name and salary, and how much less than the manager
the employee earns, Alice included.


In [19]:
# your code here


**4.** From a `CROSS JOIN`, print the pairs of employee and department in which the employee does not
work, Charlie's pairs included, and count them.


In [20]:
# your code here


**5.** Add Dana to Marketing, managed by Bob, with a salary of 58,000. Print the full join of
employees and departments in a form that runs on any version of SQLite, then roll the insert back.


In [21]:
# your code here


**6.** For every department, Marketing included, print the number of employees and the highest salary
among them.


In [22]:
# your code here


## Common errors

### sqlite3.OperationalError: ambiguous column name: Employees.Name


In [23]:
conn.execute("""
    SELECT Employees.Name
    FROM Employees
    JOIN Employees ON Employees.EmpID = Employees.ManagerID
""")


OperationalError: ambiguous column name: Employees.Name

Both copies of `Employees` are called `Employees`, so `Employees.Name` could mean either, and SQLite
cannot tell the employee from the manager. Give every copy an alias, and name every column through
one:


In [24]:
show("""
    SELECT e.Name AS employee, m.Name AS manager
    FROM Employees AS e
    JOIN Employees AS m ON m.EmpID = e.ManagerID
    ORDER BY e.Name
""")


employee  manager
--------  -------
Bob       Alice
Charlie   Alice
(2 rows)


### sqlite3.OperationalError: no such column: Departments.DeptName


In [25]:
conn.execute("""
    SELECT e.Name, Departments.DeptName
    FROM Employees AS e
    JOIN Departments AS d ON d.DeptID = e.DeptID
""")


OperationalError: no such column: Departments.DeptName

Once a table has an alias, the alias is its only name in that query, so `Departments.DeptName` names
a table the query does not have. Use the alias:


In [26]:
show("""
    SELECT e.Name, d.DeptName
    FROM Employees AS e
    JOIN Departments AS d ON d.DeptID = e.DeptID
    ORDER BY e.Name
""")


Name   DeptName
-----  --------
Alice  HR
Bob    IT
(2 rows)


### No error, and nine rows: a JOIN with no ON


In [27]:
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    JOIN Departments
    ORDER BY Employees.Name, Departments.DeptID
""")


Name     DeptName
-------  ---------
Alice    HR
Alice    IT
Alice    Marketing
Bob      HR
Bob      IT
Bob      Marketing
Charlie  HR
Charlie  IT
Charlie  Marketing
(9 rows)


The `ON` was left out, and SQLite ran the join anyway, as a cross join: every employee beside every
department, nine rows that read like real assignments. PostgreSQL refuses a `JOIN` with no `ON`, and
SQLite does not, so check a join's row count against what the tables can match. Put the condition
back:


In [28]:
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    JOIN Departments ON Employees.DeptID = Departments.DeptID
    ORDER BY Employees.Name
""")


Name   DeptName
-----  --------
Alice  HR
Bob    IT
(2 rows)


### No error, and one person in Marketing: COUNT(*) after a left join


In [29]:
show("""
    SELECT Departments.DeptName, COUNT(*) AS people
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    GROUP BY Departments.DeptID
    ORDER BY Departments.DeptID
""")


DeptName   people
---------  ------
HR         1
IT         1
Marketing  1
(3 rows)


Marketing has no employees, and the query counted one. The left join gave Marketing a row, with
`NULL` in every column from `Employees`, and `COUNT(*)` counts rows. Count a column of the joined
table that is never `NULL` in a real row, such as its primary key:


In [30]:
show("""
    SELECT Departments.DeptName, COUNT(Employees.EmpID) AS people
    FROM Departments
    LEFT JOIN Employees ON Employees.DeptID = Departments.DeptID
    GROUP BY Departments.DeptID
    ORDER BY Departments.DeptID
""")


DeptName   people
---------  ------
HR         1
IT         1
Marketing  0
(3 rows)


### No error, and a salary counted twice: a join that repeats a row


In [31]:
managers_pay = conn.execute("""
    SELECT SUM(m.Salary)
    FROM Employees AS e
    JOIN Employees AS m ON m.EmpID = e.ManagerID
""").fetchone()[0]
print("what the managers earn:", managers_pay)


what the managers earn: 144000


Alice is the only manager, on 72,000, and the query found 144,000. The join made a row for every
employee with a manager, Bob and Charlie, and both rows hold Alice as `m`, so `SUM` added Alice's
salary twice. A join repeats a row once for every match. Ask which employees are managers without
joining, so that no one can be repeated:


In [32]:
managers_pay = conn.execute("""
    SELECT SUM(Salary)
    FROM Employees
    WHERE EmpID IN (SELECT ManagerID FROM Employees)
""").fetchone()[0]
print("what the managers earn:", managers_pay)


what the managers earn: 72000


### No error, and no department at all: NOT IN beside NULL


In [33]:
show("""
    SELECT DeptName
    FROM Departments
    WHERE DeptID NOT IN (SELECT DeptID FROM Employees)
    ORDER BY DeptName
""")


DeptName
--------
(0 rows)


Marketing has no employees, and the query found no department at all. The list that `NOT IN`
compares with is 1, 2 and Charlie's `NULL`, and `3 NOT IN (1, 2, NULL)` cannot be true, since 3 might
equal the unknown value: the answer is `NULL`, and `WHERE` keeps only true. `NOT EXISTS` looks for a
matching row instead, and a `NULL` simply matches nothing:


In [34]:
show("""
    SELECT DeptName
    FROM Departments
    WHERE NOT EXISTS (SELECT 1 FROM Employees WHERE Employees.DeptID = Departments.DeptID)
    ORDER BY DeptName
""")


DeptName
---------
Marketing
(1 row)


### No error, and every employee kept: a condition on the left table in ON


In [35]:
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    LEFT JOIN Departments ON Employees.DeptID = Departments.DeptID AND Employees.Name = 'Alice'
    ORDER BY Employees.Name
""")


Name     DeptName
-------  --------
Alice    HR
Bob      NULL
Charlie  NULL
(3 rows)


The query was meant to find Alice's department, and returned all three employees, with Bob's
department gone. A left join keeps every row of the left table whatever `ON` says, so a condition in
`ON` only decides which rows of the right table match. A condition that chooses rows of the left
table belongs in `WHERE`:


In [36]:
show("""
    SELECT Employees.Name, Departments.DeptName
    FROM Employees
    LEFT JOIN Departments ON Employees.DeptID = Departments.DeptID
    WHERE Employees.Name = 'Alice'
    ORDER BY Employees.Name
""")


Name   DeptName
-----  --------
Alice  HR
(1 row)


Last, close the connection, which also discards the database, since it was only ever in memory:


In [37]:
conn.close()


## Recap

- An inner join returns only the pairs of rows for which `ON` is true, and `JOIN` alone means
  `INNER JOIN`.
- A left join adds the left table's unmatched rows, a right join the right table's, and a full join
  both, with `NULL` in the other table's columns.
- `A RIGHT JOIN B` returns the rows of `B LEFT JOIN A`, and a full join can be written as a left
  join, `UNION ALL`, and a second left join that keeps only unmatched rows, for SQLite before 3.39.0
  and for MySQL.
- A cross join returns every pair, and so does a `JOIN` with its `ON` left out, which SQLite allows.
- `NULL` matches nothing, so a row with a `NULL` in its join column is unmatched in every join.
- A self join needs an alias for every copy of the table, and once a table has an alias, only the
  alias names it.
- In an outer join, a condition in `ON` decides what matches, and a condition in `WHERE` removes rows
  afterwards.
- A join repeats a row once for every match, so count a column of the joined table, not rows, and sum
  without joining when a row could match twice. Find rows with no match with `NOT EXISTS`, never
  `NOT IN`.


## What is next

The **Parameters** notebook looks at the `?` in so many of this guide's queries: why a value goes in
through a placeholder instead of being written into the SQL, and what an f-string lets a stranger do
to a query.


---

&#8592; **Previous:** [SQL Syntax](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/04-sql-syntax.ipynb)  &nbsp;·&nbsp;  [sqlite3, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlite3-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Parameters](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlite3-deep-dive/06-parameters.ipynb) &#8594;
